# Databricks Release Notes ETL

Fetches **AWS**, **Azure**, and **GCP** Databricks release notes from:
- **RSS Feeds**: AWS, Azure, and GCP Databricks documentation
- **What's Coming pages**: Upcoming features from all three cloud platforms

Data is upserted into the target Delta table (configured via `catalog` and `schema` widget parameters) via MERGE with content-hash change detection. Each run reports **inserted**, **updated**, and **untouched** entry counts with detail tables.

In [0]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import re
import hashlib
from datetime import datetime, date

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DateType
from pyspark.sql.functions import (
    col, current_timestamp, regexp_replace, trim,
    when, row_number, lit, md5, concat_ws, coalesce,
)
from pyspark.sql.window import Window

# ── Configuration ──────────────────────────────────────────────
# ── Widget parameters (set by DAB job or interactively) ───────
dbutils.widgets.text("catalog", "main", "Catalog Name")
dbutils.widgets.text("schema", "databricks_release_notes", "Schema Name")

TARGET_TABLE = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}.release_notes"

SOURCES = {
    "AWS_FEED":           "https://docs.databricks.com/aws/en/feed.xml",
    "AZURE_FEED":         "https://learn.microsoft.com/en-us/azure/databricks/feed.xml",
    "GCP_FEED":           "https://docs.databricks.com/gcp/en/feed.xml",
    "AWS_WHATS_COMING":   "https://docs.databricks.com/aws/en/release-notes/whats-coming",
    "AZURE_WHATS_COMING": "https://learn.microsoft.com/en-gb/azure/databricks/release-notes/whats-coming",
    "GCP_WHATS_COMING":   "https://docs.databricks.com/gcp/en/release-notes/whats-coming",
}

# Columns whose content determines whether a row has changed
CONTENT_COLS = [
    "cloud", "title", "description", "url",
    "status", "feature", "impact", "categories", "source",
]

# ── HTTP session with automatic retry ─────────────────────────
def _build_http_session(retries: int = 3, backoff: float = 1.0) -> requests.Session:
    """Create an HTTP session with exponential-backoff retries."""
    session = requests.Session()
    adapter = HTTPAdapter(
        max_retries=Retry(
            total=retries,
            backoff_factor=backoff,
            status_forcelist=[429, 500, 502, 503, 504],
        )
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

http = _build_http_session()

print("✅ Configuration loaded")
for key, url in SOURCES.items():
    print(f"   {key}: {url}")
print(f"   TARGET_TABLE: {TARGET_TABLE}")

In [0]:
# ── Status classification ─────────────────────────────────────
_STATUS_PATTERNS = [
    (r"generally available|\(ga\)",                       "GA"),
    (r"public preview",                                    "Public Preview"),
    (r"private preview",                                   "Private Preview"),
    (r"\bbeta\b",                                          "Beta"),
    (r"deprecated|deprecation|retirement",                 "Deprecated"),
    (r"breaking change|behavioral change|behavior change", "Breaking Change"),
    (r"maintenance update",                                "Maintenance"),
]

def extract_status(title: str, categories: list[str]) -> str:
    """Classify release status from title keywords and RSS categories."""
    lower = title.lower()
    for pattern, status in _STATUS_PATTERNS:
        if re.search(pattern, lower):
            return status
    if "whatscoming" in [c.lower() for c in categories]:
        return "Upcoming"
    return "Released"


# ── Impact classification ─────────────────────────────────────
_HIGH_IMPACT_STATUSES = {"Breaking Change", "Deprecated"}
_LOW_IMPACT_STATUSES  = {"Public Preview", "Private Preview", "Beta", "Maintenance"}

def classify_impact(title: str, status: str) -> str:
    """Derive impact level from status and title keywords."""
    if status in _HIGH_IMPACT_STATUSES:
        return "High"
    if status == "Upcoming" and re.search(r"breaking|behavior", title, re.I):
        return "High"
    if status in ("GA", "Upcoming"):
        return "Medium"
    if status in _LOW_IMPACT_STATUSES:
        return "Low"
    return "Medium"


# ── Shared constants ──────────────────────────────────────────
META_CATEGORIES = frozenset({
    "product", "whatscoming", "aibi", "dbsql", "genie",
    "geniespaces", "databricksone", "release notes",
    "release-notes", "what's coming",
})

ZW_RE = re.compile("[\u200b\u200c\u200d\ufeff]")          # zero-width chars
SMART_PUNCT = str.maketrans({
    "\u2018": "'", "\u2019": "'",   # smart single quotes
    "\u201c": '"', "\u201d": '"',   # smart double quotes
    "\u2013": "-", "\u2014": "-",   # en/em dashes
    "\u00a0": " ",                  # non-breaking space
})
SKIP_TITLES = {"additional resources", "in this article", "next steps", "see also"}


def _clean_text(text: str, max_len: int | None = None) -> str:
    """Normalise Unicode and optionally truncate.

    Replaces zero-width characters with a space (so adjacent words stay
    separated), swaps smart punctuation for ASCII equivalents, collapses
    runs of whitespace, and trims.  When *max_len* is given the result is
    truncated to that many characters.
    """
    cleaned = ZW_RE.sub(" ", text).translate(SMART_PUNCT)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    if max_len and len(cleaned) > max_len:
        cleaned = cleaned[:max_len].rstrip()
    return cleaned

print("✅ Helper functions defined")

In [0]:
def _parse_date(raw: str) -> date:
    """Try common date formats; fall back to today."""
    s = raw.strip()
    for fmt in (
        "%a, %d %b %Y %H:%M:%S %Z",
        "%a, %d %b %Y %H:%M:%S %z",
        "%Y-%m-%dT%H:%M:%S%z",
        "%Y-%m-%d",
    ):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    try:
        return datetime.strptime(
            re.sub(r"\s+[A-Z]{2,4}$", "", s), "%a, %d %b %Y %H:%M:%S"
        ).date()
    except ValueError:
        return date.today()


def parse_rss_feed(url: str, cloud: str) -> list[dict]:
    """Fetch and parse an RSS/Atom feed into release-note dicts."""
    print(f"  Fetching RSS: {url}")
    resp = http.get(url, timeout=30)
    resp.raise_for_status()

    root    = ET.fromstring(resp.content)
    entries = []

    for item in root.findall(".//item"):
        title = _clean_text(item.findtext("title") or "")
        link  = (item.findtext("link") or "").strip()
        guid  = (item.findtext("guid") or link).strip()
        desc  = (item.findtext("description") or "").strip()

        pub_date    = _parse_date(item.findtext("pubDate") or "")
        description = _clean_text(
            BeautifulSoup(desc, "html.parser").get_text(separator=" ", strip=True),
            max_len=2000,
        )

        categories = [c.text.strip() for c in item.findall("category") if c.text]
        candidates = [c for c in categories if c.lower() not in META_CATEGORIES]
        feature    = candidates[0] if candidates else (categories[0] if categories else "General")

        status = extract_status(title, categories)
        impact = classify_impact(title, status)

        entries.append(dict(
            id          = hashlib.md5(f"{cloud}:{guid}".encode()).hexdigest(),
            cloud       = cloud,
            title       = title,
            description = description,
            pub_date    = pub_date,
            url         = link,
            status      = status,
            feature     = feature,
            impact      = impact,
            categories  = ", ".join(categories),
            source      = "rss_feed",
        ))

    print(f"    → {len(entries)} entries")
    return entries


print("✅ RSS parser defined")

In [0]:
def scrape_whats_coming(url: str, cloud: str, existing_titles: set[str]) -> list[dict]:
    """Scrape the What's Coming page for features not already in the RSS feed."""
    print(f"  Scraping: {url}")
    resp = http.get(url, timeout=30)
    resp.raise_for_status()

    soup    = BeautifulSoup(resp.content, "html.parser")
    content = (
        soup.select_one("div.theme-doc-markdown")
        or soup.select_one("main")
        or soup
    )
    entries = []
    today   = date.today()

    for h2 in content.find_all("h2"):
        for a in h2.find_all("a", class_="hash-link"):
            a.decompose()
        title = _clean_text(h2.get_text(separator=" ", strip=True))
        if len(title) < 20 or title.lower() in SKIP_TITLES or title in existing_titles:
            continue

        desc_parts = []
        sib = h2.find_next_sibling()
        while sib and sib.name != "h2":
            if sib.name in ("p", "li", "ul", "ol"):
                text = sib.get_text(separator=" ", strip=True)
                if text:
                    desc_parts.append(text)
            sib = sib.find_next_sibling()

        anchor     = h2.get("id", "")
        categories = ["whatscoming"]
        status     = extract_status(title, categories)
        impact     = classify_impact(title, status)

        entries.append(dict(
            id          = hashlib.md5(f"{cloud}:whatscoming:{title}".encode()).hexdigest(),
            cloud       = cloud,
            title       = title,
            description = _clean_text(" ".join(desc_parts), max_len=2000),
            pub_date    = today,
            url         = f"{url}#{anchor}" if anchor else url,
            status      = status,
            feature     = "Upcoming",
            impact      = impact,
            categories  = "whatscoming",
            source      = "whats_coming_page",
        ))

    print(f"    → {len(entries)} entries")
    return entries


print("✅ What's Coming scraper defined")

In [0]:
# ── Fetch all sources ─────────────────────────────────────────
print("Fetching RSS feeds …")
aws_rss   = parse_rss_feed(SOURCES["AWS_FEED"],   "AWS")
azure_rss = parse_rss_feed(SOURCES["AZURE_FEED"], "Azure")
gcp_rss   = parse_rss_feed(SOURCES["GCP_FEED"],   "GCP")

print("\nScraping What's Coming pages …")
aws_upcoming   = scrape_whats_coming(SOURCES["AWS_WHATS_COMING"],   "AWS",   {e["title"] for e in aws_rss})
azure_upcoming = scrape_whats_coming(SOURCES["AZURE_WHATS_COMING"], "Azure", {e["title"] for e in azure_rss})
gcp_upcoming   = scrape_whats_coming(SOURCES["GCP_WHATS_COMING"],   "GCP",   {e["title"] for e in gcp_rss})

all_entries = aws_rss + azure_rss + gcp_rss + aws_upcoming + azure_upcoming + gcp_upcoming
print(f"\nTotal raw entries: {len(all_entries)}")

# ── Build Spark DataFrame ─────────────────────────────────────
schema = StructType([
    StructField("id",          StringType(), False),
    StructField("cloud",       StringType(), False),
    StructField("title",       StringType(), False),
    StructField("description", StringType(), True),
    StructField("pub_date",    DateType(),   True),
    StructField("url",         StringType(), True),
    StructField("status",      StringType(), True),
    StructField("feature",     StringType(), True),
    StructField("impact",      StringType(), True),
    StructField("categories",  StringType(), True),
    StructField("source",      StringType(), True),
])

staged_df = spark.createDataFrame([Row(**e) for e in all_entries], schema=schema)
staged_df = staged_df.withColumn("ingested_at", current_timestamp())

# Defence-in-depth: replace any zero-width chars that survived Python cleaning
staged_df = staged_df.withColumn(
    "title",
    trim(regexp_replace(
        regexp_replace(col("title"), "[\u200b\u200c\u200d\ufeff]", " "),
        "\\s+", " ",
    )),
)

# Drop junk rows
staged_df = (
    staged_df
    .filter(col("title").isNotNull() & (col("title") != ""))
    .filter(~col("title").isin("Additional resources", "In this article", "Next steps", "See also"))
)

# Reclassify breaking/behavioral changes that slipped through as 'Released'
staged_df = (
    staged_df
    .withColumn("status", when(
        col("title").rlike("(?i)breaking change|behavioral change|behavior change")
        & (col("status") == "Released"),
        lit("Breaking Change"),
    ).otherwise(col("status")))
    .withColumn("impact", when(
        col("status") == "Breaking Change", lit("High")
    ).otherwise(col("impact")))
)

# Deduplicate: prefer rss_feed over whats_coming_page, then newest
dedup_window = Window.partitionBy("title", "cloud").orderBy(
    when(col("source") == "rss_feed", 0).otherwise(1).asc(),
    col("pub_date").desc(),
)
staged_df = (
    staged_df
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# Content hash for change detection during MERGE
staged_df = staged_df.withColumn(
    "content_hash",
    md5(concat_ws("|", *[coalesce(col(c), lit("")) for c in CONTENT_COLS])),
)

staged_df.createOrReplaceTempView("release_notes_staged")
staged_count = staged_df.count()
print(f"\n✅ Staged {staged_count} deduplicated entries")

In [0]:
# ── Ensure target schema and table exist ──────────────────────
_catalog = dbutils.widgets.get("catalog")
_schema = dbutils.widgets.get("schema")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{_catalog}`.`{_schema}`")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        id          STRING NOT NULL,
        cloud       STRING NOT NULL,
        title       STRING NOT NULL,
        description STRING,
        pub_date    DATE,
        url         STRING,
        status      STRING,
        feature     STRING,
        impact      STRING,
        categories  STRING,
        source      STRING,
        ingested_at TIMESTAMP
    ) USING DELTA
""")

# ── Pre-classify staged rows ─────────────────────────────────
DISPLAY_COLS = ["cloud", "title", "status", "feature", "impact", "pub_date", "source"]

target_hashed = (
    spark.table(TARGET_TABLE)
    .withColumn("t_hash", md5(concat_ws("|", *[coalesce(col(c), lit("")) for c in CONTENT_COLS])))
    .select(col("id").alias("t_id"), "t_hash")
)

staged = spark.table("release_notes_staged")
comparison = staged.alias("s").join(target_hashed, col("s.id") == col("t_id"), "left")

# New rows (no match in target)
inserted_df = (
    comparison.filter(col("t_hash").isNull())
    .select("s.*")
    .drop("content_hash")
)

# Changed rows (match exists, content differs)
updated_df = (
    comparison.filter(col("t_hash").isNotNull() & (col("s.content_hash") != col("t_hash")))
    .select("s.*")
    .drop("content_hash")
)

# Materialize BEFORE the merge (temp views are lazy and would re-evaluate
# against the post-merge target, showing 0 rows)
inserted_pdf = inserted_df.select(*DISPLAY_COLS).orderBy(col("pub_date").desc()).toPandas()
updated_pdf  = updated_df.select(*DISPLAY_COLS).orderBy(col("pub_date").desc()).toPandas()

n_inserted  = len(inserted_pdf)
n_updated   = len(updated_pdf)
rows_before = spark.table(TARGET_TABLE).count()
n_untouched = rows_before - n_updated

# ── MERGE with conditional update (skip unchanged rows) ─────
spark.sql(f"""
    MERGE INTO {TARGET_TABLE} AS tgt
    USING release_notes_staged AS src
    ON tgt.id = src.id
    WHEN MATCHED
        AND md5(concat_ws('|',
            coalesce(tgt.cloud,''),       coalesce(tgt.title,''),
            coalesce(tgt.description,''), coalesce(tgt.url,''),
            coalesce(tgt.status,''),      coalesce(tgt.feature,''),
            coalesce(tgt.impact,''),      coalesce(tgt.categories,''),
            coalesce(tgt.source,'')
        )) != src.content_hash
    THEN UPDATE SET
        tgt.cloud       = src.cloud,
        tgt.title       = src.title,
        tgt.description = src.description,
        tgt.pub_date    = src.pub_date,
        tgt.url         = src.url,
        tgt.status      = src.status,
        tgt.feature     = src.feature,
        tgt.impact      = src.impact,
        tgt.categories  = src.categories,
        tgt.source      = src.source,
        tgt.ingested_at = src.ingested_at
    WHEN NOT MATCHED THEN INSERT (
        id, cloud, title, description, pub_date,
        url, status, feature, impact, categories,
        source, ingested_at
    ) VALUES (
        src.id, src.cloud, src.title, src.description, src.pub_date,
        src.url, src.status, src.feature, src.impact, src.categories,
        src.source, src.ingested_at
    )
""")

rows_after = spark.table(TARGET_TABLE).count()

# ── Summary ─────────────────────────────────────────────────
sep = "=" * 60
dash = "─" * 29
print(sep)
print("  MERGE RESULTS")
print(sep)
print(f"  Rows before:   {rows_before:>6,}")
print(f"  Rows after:    {rows_after:>6,}")
print(f"  {dash}")
print(f"  Inserted:      {n_inserted:>6,}")
print(f"  Updated:       {n_updated:>6,}")
print(f"  Untouched:     {n_untouched:>6,}")
print(sep)

In [0]:
print(f"🔄 Updated entries: {len(updated_pdf)}")
if len(updated_pdf) > 0:
    display(spark.createDataFrame(updated_pdf))
else:
    print("No entries were updated in this run.")

In [0]:
print(f"🆕 Inserted entries: {len(inserted_pdf)}")
if len(inserted_pdf) > 0:
    display(spark.createDataFrame(inserted_pdf))
else:
    print("No new entries were inserted in this run.")